# NurseGemma - Simple Medical AI Assistant

**Built by a nurse, for nurses and families.**

Two simple features:
1. **Ask Questions** - Get medical explanations anyone can understand
2. **Analyze Images** - Upload wounds or scans for professional nursing documentation

---

*Powered by Google MedGemma 1.5 | MedGemma Impact Challenge Submission*

In [ ]:
# Install dependencies
!pip install -q transformers accelerate pillow requests

In [ ]:
# Setup
import torch
import requests
from PIL import Image
from io import BytesIO
from transformers import AutoProcessor, AutoModelForImageTextToText
from huggingface_hub import login

# Check GPU
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Authenticate with HuggingFace
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    login(token=secrets.get_secret('HF_TOKEN'))
    print("Authenticated via Kaggle Secrets")
except:
    login()  # Interactive login for Colab/local
    print("Authenticated interactively")

In [ ]:
# Load MedGemma 1.5 4B
MODEL_ID = "google/medgemma-1.5-4b-it"

print("Loading MedGemma 1.5...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
print(f"MedGemma loaded on {next(model.parameters()).device}")

---
## Feature 1: Ask Medical Questions

Get explanations that families can understand. No medical jargon.

In [ ]:
def ask_nursegemma(question: str) -> str:
    """
    Ask a medical question and get a family-friendly explanation.
    """
    prompt = f"""You are NurseGemma, a friendly nurse educator helping families understand medical information.

RULES:
- Use simple words (8th grade reading level)
- Avoid medical jargon - if you must use a term, explain it
- Be warm and reassuring
- Use helpful analogies when possible
- Keep answers concise but complete

QUESTION: {question}

ANSWER:"""
    
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)
    
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=500, do_sample=False)
    
    response = processor.decode(output[0], skip_special_tokens=True)
    # Extract just the answer part
    if "ANSWER:" in response:
        response = response.split("ANSWER:")[-1].strip()
    return response

In [ ]:
# Example: Ask about a diagnosis
question = "What is CHF? My dad was just diagnosed and I'm scared."

print("Question:", question)
print("\n" + "="*50 + "\n")
print(ask_nursegemma(question))

In [ ]:
# Example: Ask about a medication
question = "Why does my mom take Lasix? What should we watch for?"

print("Question:", question)
print("\n" + "="*50 + "\n")
print(ask_nursegemma(question))

In [ ]:
# Example: Ask about a procedure
question = "My husband needs a cardiac catheterization. What happens during this test?"

print("Question:", question)
print("\n" + "="*50 + "\n")
print(ask_nursegemma(question))

---
## Feature 2: Medical Image Analysis

Upload wound photos or scans. Get professional nursing documentation.

In [ ]:
def analyze_wound(image_source, patient_context: str = "") -> str:
    """
    Analyze a wound image and generate nursing documentation.
    
    image_source: URL string or PIL Image
    patient_context: Optional context (e.g., "diabetic patient, sacral area")
    """
    # Load image
    if isinstance(image_source, str):
        response = requests.get(image_source, headers={"User-Agent": "NurseGemma"}, timeout=30)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = image_source
    
    context_line = f"\nPatient context: {patient_context}" if patient_context else ""
    
    prompt = f"""You are an experienced wound care nurse documenting a wound assessment.
{context_line}

Provide professional nursing documentation including:

1. WOUND TYPE & LOCATION
2. WOUND MEASUREMENTS (estimate from image)
3. WOUND BED CHARACTERISTICS
   - Tissue types (granulation, slough, eschar, epithelial)
   - Color and percentage
4. WOUND EDGES & PERIWOUND SKIN
5. DRAINAGE (if visible)
6. STAGING (for pressure injuries: Stage 1/2/3/4/Unstageable/DTPI)
7. RECOMMENDED INTERVENTIONS
8. FOLLOW-UP RECOMMENDATIONS

Use professional nursing terminology suitable for medical charting."""
    
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]
    }]
    
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)
    
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=800, do_sample=False)
    
    return processor.decode(output[0], skip_special_tokens=True)

In [ ]:
def analyze_scan(image_source, scan_type: str = "chest X-ray") -> str:
    """
    Analyze a medical scan and explain findings.
    
    image_source: URL string or PIL Image
    scan_type: Type of scan (e.g., "chest X-ray", "CT head", "MRI brain")
    """
    # Load image
    if isinstance(image_source, str):
        response = requests.get(image_source, headers={"User-Agent": "NurseGemma"}, timeout=30)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = image_source
    
    prompt = f"""You are an experienced nurse educator explaining a {scan_type} to help with clinical understanding.

Provide:

1. SCAN TYPE & QUALITY
2. KEY FINDINGS
   - Normal structures
   - Abnormalities (if any)
3. CLINICAL SIGNIFICANCE
   - What these findings might mean
4. NURSING IMPLICATIONS
   - What to monitor
   - When to notify the provider
5. PATIENT/FAMILY EXPLANATION
   - Simple explanation for the patient

Be thorough but clear."""
    
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]
    }]
    
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)
    
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=800, do_sample=False)
    
    return processor.decode(output[0], skip_special_tokens=True)

In [ ]:
# Example: Analyze a wound image
WOUND_URL = "https://upload.wikimedia.org/wikipedia/commons/f/fc/Grade3.jpg"

print("WOUND ASSESSMENT")
print("="*50)
print(analyze_wound(WOUND_URL, patient_context="elderly patient, sacral area"))

In [ ]:
# Example: Analyze a chest X-ray
CXR_URL = "https://upload.wikimedia.org/wikipedia/commons/8/87/X-ray_of_lobar_pneumonia.jpg"

print("CHEST X-RAY ANALYSIS")
print("="*50)
print(analyze_scan(CXR_URL, scan_type="chest X-ray"))

---
## Interactive Mode

Use these functions with your own questions and images!

In [ ]:
# YOUR TURN: Ask your own question
my_question = "What does it mean when they say my blood pressure is high?"

print("Your Question:", my_question)
print("\n" + "="*50 + "\n")
print(ask_nursegemma(my_question))

In [ ]:
# YOUR TURN: Analyze your own image
# Replace with your own image URL or upload an image
my_image_url = "YOUR_IMAGE_URL_HERE"

# Uncomment the type of analysis you need:
# print(analyze_wound(my_image_url, patient_context="describe patient"))
# print(analyze_scan(my_image_url, scan_type="chest X-ray"))

---

## About NurseGemma

As an ICU nurse, I spend 40% of my shift documenting instead of caring for patients. NurseGemma is the AI companion I wish I had - one that helps families understand what's happening and helps nurses document efficiently.

**Disclaimer**: NurseGemma is an educational tool. All outputs should be verified by qualified healthcare professionals. Not for diagnostic or treatment decisions.

---

*Built for the MedGemma Impact Challenge 2026*